# Walmart Store Sales Analysis & Forecasting
**Course-End Project | Applied Data Science**

This notebook analyzes Walmart weekly sales, compares store performance, examines holiday and seasonal patterns, explores economic indicators, and compares machine-learning models for Store 1 sales prediction.

**Dataset required:** `Walmart_Store_sales.csv` (place it in the same folder as this notebook).

## Objectives
- Find the store with the highest total sales.
- Compare store sales variability using standard deviation and coefficient of variation.
- Identify stores with positive Q3 2012 growth compared with Q2 2012.
- Compare holiday sales with non-holiday sales.
- Summarize monthly and semester-wise sales.
- Explore correlations between sales and economic indicators.
- Train and compare models to predict Store 1 weekly sales.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load and inspect the dataset

In [ ]:
DATA_PATH = "Walmart_Store_sales.csv"  # Update if your CSV is elsewhere

df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df.info()

print("\\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("\\nDuplicate rows:", df.duplicated().sum())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

## 3. Prepare date fields

In [ ]:
df["Holiday_Flag"] = pd.to_numeric(df["Holiday_Flag"], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%B")
df["Quarter"] = df["Date"].dt.quarter
df["Semester"] = np.where(df["Month"] <= 6, "Semester 1", "Semester 2")

display(df.describe().T)

## 4. Store-wise sales analysis

### 4.1 Store with the highest total sales

In [ ]:
store_sales = (
    df.groupby("Store", as_index=False)["Weekly_Sales"]
      .sum()
      .sort_values("Weekly_Sales", ascending=False)
)
display(store_sales)

top_store = store_sales.iloc[0]
print(f"Highest total sales: Store {int(top_store['Store'])} — ${top_store['Weekly_Sales']:,.2f}")

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=store_sales, x="Store", y="Weekly_Sales", color="steelblue")
plt.title("Total Sales by Store")
plt.xlabel("Store")
plt.ylabel("Total weekly sales")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 4.2 Standard deviation and coefficient of variation

In [ ]:
store_variability = (
    df.groupby("Store")["Weekly_Sales"]
      .agg(Mean_Sales="mean", Std_Dev="std")
      .reset_index()
)
store_variability["CV_Percent"] = (
    store_variability["Std_Dev"] / store_variability["Mean_Sales"] * 100
)
display(store_variability.sort_values("Std_Dev", ascending=False))

highest_std_store = store_variability.loc[store_variability["Std_Dev"].idxmax()]
highest_cv_store = store_variability.loc[store_variability["CV_Percent"].idxmax()]

print(f"Highest standard deviation: Store {int(highest_std_store['Store'])}")
print(f"Highest coefficient of variation: Store {int(highest_cv_store['Store'])}")

## 5. Q2 vs Q3 sales growth in 2012

In [ ]:
quarterly_2012 = (
    df[df["Year"] == 2012]
      .groupby(["Store", "Quarter"])["Weekly_Sales"]
      .sum()
      .unstack(fill_value=0)
)

# Q2 is quarter 2; Q3 is quarter 3.
quarterly_2012["Growth_Percent"] = np.where(
    quarterly_2012[2] != 0,
    (quarterly_2012[3] - quarterly_2012[2]) / quarterly_2012[2] * 100,
    np.nan
)

positive_growth = quarterly_2012[
    quarterly_2012["Growth_Percent"] > 0
].sort_values("Growth_Percent", ascending=False)

display(quarterly_2012)
print("Stores with positive Q3 growth over Q2:")
display(positive_growth[["Growth_Percent"]])

## 6. Holiday sales analysis

In [ ]:
holiday_summary = (
    df.groupby("Holiday_Flag")["Weekly_Sales"]
      .agg(["mean", "median", "count"])
      .rename(index={0: "Non-holiday", 1: "Holiday"})
)
display(holiday_summary)

non_holiday_average = df.loc[df["Holiday_Flag"] == 0, "Weekly_Sales"].mean()

holiday_date_sales = (
    df[df["Holiday_Flag"] == 1]
      .groupby("Date")["Weekly_Sales"]
      .mean()
      .rename("Average_Store_Sales")
      .reset_index()
)

above_average_holidays = holiday_date_sales[
    holiday_date_sales["Average_Store_Sales"] > non_holiday_average
].sort_values("Average_Store_Sales", ascending=False)

print(f"Average non-holiday weekly sales per store: ${non_holiday_average:,.2f}")
display(above_average_holidays)

In [ ]:
plot_data = df.copy()
plot_data["Day_Type"] = plot_data["Holiday_Flag"].map({0: "Non-holiday", 1: "Holiday"})

plt.figure(figsize=(7, 5))
sns.boxplot(data=plot_data, x="Day_Type", y="Weekly_Sales")
plt.title("Weekly Sales: Holiday vs Non-holiday")
plt.xlabel("")
plt.tight_layout()
plt.show()

## 7. Monthly and semester-wise sales

In [ ]:
monthly_sales = (
    df.groupby(["Year", "Month", "Month_Name"], as_index=False)["Weekly_Sales"]
      .sum()
      .sort_values(["Year", "Month"])
)

semester_sales = (
    df.groupby(["Year", "Semester"], as_index=False)["Weekly_Sales"]
      .sum()
)

display(monthly_sales)
display(semester_sales)

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="Month_Name", y="Weekly_Sales", hue="Year", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total weekly sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=semester_sales, x="Year", y="Weekly_Sales", hue="Semester")
plt.title("Semester-wise Sales")
plt.ylabel("Total weekly sales")
plt.tight_layout()
plt.show()

In [ ]:
highest_month = monthly_sales.loc[monthly_sales["Weekly_Sales"].idxmax()]
lowest_month = monthly_sales.loc[monthly_sales["Weekly_Sales"].idxmin()]
highest_semester = semester_sales.loc[semester_sales["Weekly_Sales"].idxmax()]

print(f"Highest-sales month: {highest_month['Month_Name']} {int(highest_month['Year'])}")
print(f"Lowest-sales month: {lowest_month['Month_Name']} {int(lowest_month['Year'])}")
print(f"Highest-sales semester: {highest_semester['Semester']} {int(highest_semester['Year'])}")

## 8. Relationship between sales and economic factors

Correlation measures linear association; it does not prove that one variable causes another.

In [ ]:
economic_columns = ["Weekly_Sales", "CPI", "Unemployment", "Fuel_Price"]
economic_corr = df[economic_columns].corr()
display(economic_corr)

plt.figure(figsize=(7, 5))
sns.heatmap(economic_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Sales and Economic Factor Correlations")
plt.tight_layout()
plt.show()

## 9. Store 1 sales prediction

### 9.1 Prepare the data

The date is converted to a numeric day count from Store 1's first recorded date. A chronological 80/20 split is used so that the latest observations are held out for testing.

In [ ]:
store_1 = df[df["Store"] == 1].copy().sort_values("Date").reset_index(drop=True)
store_1["Day"] = (store_1["Date"] - store_1["Date"].min()).dt.days

features = [
    "Day", "Holiday_Flag", "Temperature",
    "Fuel_Price", "CPI", "Unemployment"
]
target = "Weekly_Sales"

model_data = store_1[features + [target]].dropna().reset_index(drop=True)

split_index = int(len(model_data) * 0.8)
X = model_data[features]
y = model_data[target]

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

### 9.2 Train and compare models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    )
}

trained_models = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

model_results = pd.DataFrame(results).sort_values("RMSE")
display(model_results)

### 9.3 Best model and prediction comparison

In [ ]:
best_model_name = model_results.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model based on lowest test RMSE:", best_model_name)

prediction_comparison = pd.DataFrame({
    "Actual_Sales": y_test.to_numpy(),
    "Predicted_Sales": best_predictions
})
display(prediction_comparison.head(10))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(range(len(y_test)), y_test.to_numpy(), label="Actual sales", marker="o")
plt.plot(range(len(best_predictions)), best_predictions, label="Predicted sales", marker="o")
plt.title(f"Store 1: Actual vs Predicted Sales — {best_model_name}")
plt.xlabel("Test observation (chronological order)")
plt.ylabel("Weekly sales")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Summary of key results

In [ ]:
print("WALMART STORE SALES ANALYSIS — SUMMARY")
print("-" * 48)
print(f"Store with highest total sales: Store {int(top_store['Store'])}")
print(f"Store with highest sales standard deviation: Store {int(highest_std_store['Store'])}")
print(f"Store with highest coefficient of variation: Store {int(highest_cv_store['Store'])}")
print(f"Number of stores with positive Q3 vs Q2 2012 growth: {len(positive_growth)}")
print(f"Holiday dates above the non-holiday average: {len(above_average_holidays)}")
print(f"Best Store 1 model by RMSE: {best_model_name}")

## 11. Conclusion

This course-end project applies Python-based data analysis and machine-learning techniques to Walmart weekly sales data. The analysis covers store performance, sales variability, quarterly growth, holiday comparisons, monthly and semester trends, economic-factor correlations, and Store 1 sales prediction.

The models are compared using a chronological holdout set. The reported metrics describe performance on this test period and should not be interpreted as a guarantee of future performance.

## 12. Possible improvements

- Tune model hyperparameters using time-aware cross-validation.
- Explore dedicated time-series forecasting methods.
- Add lagged sales and rolling-average features.
- Investigate store-specific and holiday-type effects.
- Build an interactive dashboard to present the findings.

---

**Project Type:** Course-End Project  
**Domain:** Applied Data Science | Retail Analytics | Machine Learning